### Day 11 · OOP 组合 + Pythonic 对象协议 (L4)

今天让自定义对象**像内置类型一样自然**。
你会学会:组合优于继承、运算符重载、迭代协议、
让 `len(obj)`、`obj1 + obj2`、`for x in obj` 全部可用。

> **前置知识**:Day 08 L1(类/方法/属性)、Day 09 L2(继承)、
> Day 10 L3(多态/抽象基类)。
> 如果这些还不熟,请先复习。


#### 组合 vs 继承 —— is-a 还是 has-a?

> **痛点**  
继承用多了,改一处崩一片。子类和父类**强耦合**,
父类加个参数,所有子类都要跟着改。

> **类比**  
- **is-a**:苹果 **是** 一种水果 → 用继承(Apple 继承 Fruit)
- **has-a**:订单 **有** 一个购物车 + 收货地址 → 用组合

**判断口诀**:把两个类放进"X 是 Y"和"X 有 Y"各读一遍,
通顺的就是正确关系。


**ASCII 内存图 —— 组合 vs 继承**

```
继承(强耦合):              组合(松耦合):
┌─────────────┐           ┌──────────┐
│ Order       │           │ Order    │
│  └─ 继承 Cart│           │  └─ cart ──→ ┌──────┐
│     └─ items │           │              │ Cart │
└─────────────┘           │  └─ addr ──→ │ Addr │
改 Cart 影响 Order        └──────────┘   └──────┘
                          改 Cart 不影响 Order
```


In [ ]:
# 继承写法(不推荐)
class Cart:
    def __init__(self):
        self.items = []

class Order(Cart):        # Order is-a Cart? 不通顺!
    def __init__(self, addr):
        super().__init__()
        self.addr = addr

# --- 执行过程 ---
# 第 1 行 class Cart: 创建 Cart 类对象
# 第 5 行 class Order(Cart): Order 继承 Cart
#   ① Order 拥有 Cart 的所有属性和方法
#   ② 但 Order 并不"是"购物车,语义错误
# 第 9 行 Order("北京"):
#   ① 调用 Order.__init__,super() 调用 Cart.__init__
#   ② self.items = [] 绑定到 Order 实例
#   ③ self.addr = "北京" 绑定到 Order 实例

o = Order("北京")
print(o.items)   # 输出: []
print(o.addr)    # 输出: 北京


> **逐行解剖**  
① `class Order(Cart)` 让 Order 继承 Cart 的全部  
② `super().__init__()` 调用父类构造函数  
③ 问题:Order 不是 Cart,强行继承导致语义混乱  
④ 一旦 Cart 改了构造参数,Order 必须跟着改


**常见错误**

1. **语义倒置**:`class Order(Cart)` 读作"订单是购物车",
   逻辑不通,应该用组合。
2. **过度继承**:为了复用两行代码就继承一个大类,
   结果子类拿到一堆不需要的方法。
3. **忘记 super()**:子类重写 `__init__` 后忘了调
   `super().__init__()`,父类属性全部丢失。


> 问自己:
> - "订单"和"购物车"是"是一个"还是"有一个"关系?
> - 如果购物车加了一个 `discount` 属性,
>   订单类需要跟着改吗?
> - 组合的话,订单怎么访问购物车的商品?


#### 继承的三个坏味道 —— 该用组合时用了继承

> **痛点**  
看到两个类有相似代码,第一反应就继承。
结果类层次越来越深,改底层崩上层。

> **类比**  
继承像"血缘关系"——不能改;
组合像"合作关系"——随时换人。

**三个坏味道**:
1. 子类只用了父类**一小部分**方法
2. 子类**重写**了父类大部分方法
3. 继承层次**超过两层**


In [ ]:
# 坏味道 1:子类只用父类一小部分
class Animal:
    def eat(self): print("吃")
    def fly(self): print("飞")
    def swim(self): print("游")

class Dog(Animal):
    def fly(self):
        raise NotImplementedError  # 狗不会飞!

d = Dog()
d.eat()   # 输出: 吃
# d.fly()  # 报错! 继承了不该继承的

# --- 执行过程 ---
# 第 7 行 class Dog(Animal): Dog 继承 Animal
#   ① Dog 拿到 eat/fly/swim 三个方法
#   ② 但狗不会飞,fly 只能抛异常
#   ③ 这说明 Dog 不该继承 Animal,应该拆分

print("坏味道 1:继承了不需要的方法")


In [ ]:
# 坏味道 2:子类重写父类大部分方法
class Report:
    def header(self): return "=== 报告 ==="
    def body(self): return "内容"
    def footer(self): return "=== 结束 ==="

class PDFReport(Report):
    def header(self): return "<pdf header>"   # 重写
    def body(self): return "<pdf body>"       # 重写
    def footer(self): return "<pdf footer>"   # 全重写了!

# 三个方法全重写,继承毫无意义
# 应该用组合:Report has-a Renderer

print("坏味道 2:继承只为重写,不如组合")


> **逐行解剖**  
① 坏味道 1:子类拿到不需要的方法,违反**接口隔离**  
② 坏味道 2:子类重写全部方法,继承没有复用价值  
③ 两个坏味道的共同根因:**is-a 判断错误**  
④ 正确做法:把"行为"拆成独立类,用组合注入


**常见错误**

1. **"代码复用"陷阱**:为了省几行代码就继承,
   结果耦合代价远超复用收益。
2. **"未来可能用到"**:先继承再说,以后可能用。
   结果"以后"永远不来,代码债越积越多。
3. **"两层以上继承"**:A→B→C→D,改 A 不知道
   影响谁,调试地狱。


> 问自己:
> - 如果子类只用了父类的 20% 方法,继承合适吗?
> - 重写超过一半方法,是不是该用组合?
> - 继承超过两层,改底层时你能确定影响范围吗?


#### 组合设计 —— Order has-a Cart + Payment + Address

> **痛点**  
一个订单对象需要"商品列表"+"收货地址"+"支付方式",
全塞进一个类里,又臃肿又难改。

> **类比**  
订单像一份快递单:它**有**一个地址、**有**一个包裹、
**有**一个付款方式,但订单本身不是地址也不是包裹。

**设计原则**:每个类只负责一件事,复杂对象由简单对象**组合**而成。


In [ ]:
class Address:
    """收货地址"""
    def __init__(self, city, detail):
        self.city = city
        self.detail = detail

class Payment:
    """支付方式"""
    def __init__(self, method):
        self.method = method  # "微信"/"支付宝"

class Cart:
    """购物车"""
    def __init__(self):
        self.items = []

class Order:
    """订单:组合 Address + Payment + Cart"""
    def __init__(self, addr, pay, cart):
        self.addr = addr      # has-a Address
        self.pay = pay        # has-a Payment
        self.cart = cart      # has-a Cart

# --- 执行过程 ---
# 第 19 行 addr = Address("北京", "朝阳区xx路"):
#   ① 创建 Address 实例,city="北京",detail="朝阳区xx路"
# 第 20 行 pay = Payment("微信"):
#   ① 创建 Payment 实例,method="微信"
# 第 21 行 cart = Cart():
#   ① 创建 Cart 实例,items=[]
# 第 22 行 Order(addr, pay, cart):
#   ① Order 不继承任何类,只持有三个对象的引用
#   ② self.addr 指向 addr 对象(共享,不拷贝)

addr = Address("北京", "朝阳区xx路")
pay = Payment("微信")
cart = Cart()
order = Order(addr, pay, cart)
print(order.addr.city)   # 输出: 北京
print(order.pay.method)  # 输出: 微信


> **逐行解剖**  
① `self.addr = addr` 把 Address **实例**绑到 Order 上  
② Order 不复制 addr,只是**引用**它(省内存)  
③ 改 `addr.city` 会影响 `order.addr.city`(共享)  
④ 这就是组合:**持有引用**,不是**复制数据**


**ASCII 内存图**

```
order ──→ ┌─────────────────┐
          │ Order           │
          │  addr ──────→ ┌──────────┐
          │  pay  ──────→ │ Address  │
          │  cart ──────→ │ city="北京"│
          └─────────────────┘ └──────────┘
              │
              ├──────→ ┌──────────┐
              │        │ Payment  │
              │        │ method="微信"│
              │        └──────────┘
              └──────→ ┌──────────┐
                       │ Cart     │
                       │ items=[] │
                       └──────────┘
```


> 问自己:
> - Order 能继承 Address 吗? "订单是地址"通顺吗?
> - 如果支付方式要支持"信用卡",改哪里?
> - 组合后,Order 怎么访问购物车的商品?


**练习** —— 为 Order 添加 `summary()` 方法

请为 `Order` 类添加 `summary()` 方法,返回:
`"送到{city}({detail}),{N}件商品,支付方式{method}"`。


In [ ]:
# ============ 学员代码区 ============
class Address:
    def __init__(self, city, detail):
        self.city = city
        self.detail = detail

class Payment:
    def __init__(self, method):
        self.method = method

class Cart:
    def __init__(self):
        self.items = []

class Order:
    def __init__(self, addr, pay, cart):
        self.addr = addr
        self.pay = pay
        self.cart = cart
    # 请添加 summary() 方法

# addr = Address("上海", "浦东新区")
# pay = Payment("支付宝")
# cart = Cart()
# cart.items = ["书", "笔"]
# order = Order(addr, pay, cart)
# print(order.summary())


In [ ]:
# ============ 参考答案 ============
class Address:
    def __init__(self, city, detail):
        self.city = city
        self.detail = detail

class Payment:
    def __init__(self, method):
        self.method = method

class Cart:
    def __init__(self):
        self.items = []

class Order:
    def __init__(self, addr, pay, cart):
        self.addr = addr
        self.pay = pay
        self.cart = cart

    def summary(self):
        n = len(self.cart.items)
        return (f"送到{self.addr.city}"
                f"({self.addr.detail}),"
                f"{n}件商品,支付方式{self.pay.method}")

addr = Address("上海", "浦东新区")
pay = Payment("支付宝")
cart = Cart()
cart.items = ["书", "笔"]
order = Order(addr, pay, cart)
print(order.summary())
# 输出: 送到上海(浦东新区),2件商品,支付方式支付宝


#### 运算符重载本质 —— 运算符就是方法的语法糖

> **痛点**  
`cart1 + cart2` 直接报 TypeError,
因为 Python 不知道两个自定义对象怎么"加"。

> **类比**  
`+` 不是魔法,它只是调用 `__add__` 方法的**快捷方式**。
`a + b` 等价于 `a.__add__(b)`。

**核心认知**:所有运算符背后都是**双下划线方法**(魔术方法)。
你想支持什么操作,就实现对应的协议方法。


In [ ]:
# 验证:运算符就是方法
a = [1, 2]
b = [3, 4]

# 两种写法完全等价
r1 = a + b          # 语法糖
r2 = a.__add__(b)   # 实际调用

print(r1)  # 输出: [1, 2, 3, 4]
print(r2)  # 输出: [1, 2, 3, 4]
print(r1 == r2)  # 输出: True

# --- 执行过程 ---
# 第 5 行 a + b:
#   ① Python 看到 +,查找 a 的 __add__ 方法
#   ② 调用 a.__add__(b),返回新列表
#   ③ 赋值给 r1
# 第 6 行 a.__add__(b):
#   ① 直接调用 __add__ 方法
#   ② 结果和 a + b 完全一样


> **逐行解剖**  
① `a + b` 是**语法糖**,实际调用 `a.__add__(b)`  
② 列表的 `__add__` 返回**新列表**,不修改原列表  
③ 自定义类没有 `__add__`,所以 `obj1 + obj2` 报错  
④ 实现 `__add__` 后,`+` 就自动可用


**常见错误**

1. **以为 `+` 会修改原对象**:`a + b` 返回新对象,
   `a` 本身不变。要修改用 `a += b`(调用 `__iadd__`)。
2. **忘记返回结果**:`__add__` 必须 `return` 新对象,
   `None + something` 会报错。
3. **类型不匹配**:`__add__` 里没检查 `other` 类型,
   导致运行时错误。


> 问自己:
> - `a + b` 和 `a.__add__(b)` 完全等价吗?
> - `__add__` 应该返回新对象还是修改 self?
> - 如果 `other` 不是同类对象,应该怎么处理?


#### __add__ —— 合并两辆购物车

> **痛点**  
想把两辆购物车的商品合并,只能手动 `extend`。
如果 `cart1 + cart2` 就能合并,代码会自然很多。

> **类比**  
`+` 就像把两个购物车的商品**倒在一起**,
返回一辆**新**购物车,两辆原车不变。


In [ ]:
class Cart:
    def __init__(self, items=None):
        # 默认参数不用可变对象(常见坑)
        self.items = items if items else []

    def __add__(self, other):
        """合并两辆购物车,返回新购物车"""
        # 创建新列表,不修改原对象
        new_items = self.items + other.items
        return Cart(new_items)

# --- 执行过程 ---
# 第 10 行 c1 = Cart(["苹果", "香蕉"]):
#   ① items 参数 = ["苹果","香蕉"]
#   ② self.items 指向这个列表
# 第 11 行 c2 = Cart(["牛奶"]):
#   ① self.items = ["牛奶"]
# 第 12 行 c3 = c1 + c2:
#   ① Python 调用 c1.__add__(c2)
#   ② new_items = ["苹果","香蕉"] + ["牛奶"]
#   ③ 返回新 Cart 实例
#   ④ c1 和 c2 本身不变(安全语义)

c1 = Cart(["苹果", "香蕉"])
c2 = Cart(["牛奶"])
c3 = c1 + c2
print(c1.items)  # 输出: ['苹果', '香蕉']  ← 不变!
print(c2.items)  # 输出: ['牛奶']          ← 不变!
print(c3.items)  # 输出: ['苹果', '香蕉', '牛奶']


> **逐行解剖**  
① `def __add__(self, other)` 定义 `+` 行为  
② `self.items + other.items` 调用列表的 `__add__`  
③ `return Cart(new_items)` 返回**新**购物车  
④ 不修改 self 和 other,符合**不可变语义**


**ASCII 内存图**

```
c1 ──→ ┌──────┐      c2 ──→ ┌──────┐
       │ Cart │             │ Cart │
       │items │             │items │
       │ ↓    │             │ ↓    │
       │[苹果,│             │[牛奶] │
       │ 香蕉]│             └──────┘
       └──────┘
              │
              │ c1 + c2 → 调用 __add__
              ↓
       c3 ──→ ┌──────┐
              │ Cart │
              │items │
              │ ↓    │
              │[苹果,│
              │ 香蕉,│
              │ 牛奶]│
              └──────┘
```


**常见错误**

1. **修改原对象**:`self.items.extend(other.items)` 然后
   `return self` —— 这修改了 c1,违反直觉。
2. **忘记返回**:`__add__` 里只合并没 return,
   `c3` 拿到 `None`。
3. **共享引用**:`new_items = self.items` 然后 extend,
   修改 new_items 会影响原购物车。


> 问自己:
> - `__add__` 应该返回新对象还是修改 self?
> - 为什么 `self.items + other.items` 不会修改原列表?
> - 如果 `other` 不是 Cart 实例,应该怎么报错?


**练习** —— 实现 `__add__` 合并两个 `Product` 清单

定义 `ProductList` 类,支持 `list1 + list2` 合并。


In [ ]:
# ============ 学员代码区 ============
class ProductList:
    def __init__(self, products=None):
        self.products = products if products else []
    # 请实现 __add__

# p1 = ProductList(["苹果"])
# p2 = ProductList(["香蕉"])
# p3 = p1 + p2
# print(p3.products)  # 期望: ['苹果', '香蕉']


In [ ]:
# ============ 参考答案 ============
class ProductList:
    def __init__(self, products=None):
        self.products = products if products else []

    def __add__(self, other):
        # 类型检查,友好报错
        if not isinstance(other, ProductList):
            return NotImplemented
        # 返回新对象,不修改原对象
        return ProductList(self.products + other.products)

p1 = ProductList(["苹果"])
p2 = ProductList(["香蕉"])
p3 = p1 + p2
print(p3.products)  # 输出: ['苹果', '香蕉']
print(p1.products)  # 输出: ['苹果']  ← 不变!


#### __len__ —— 让 len(cart) 返回件数

> **痛点**  
`len(cart)` 报错,因为 Python 不知道自定义对象
的"长度"是什么。必须 `len(cart.items)` 才行。

> **类比**  
`len()` 像问"这箱苹果有几个?",
实现 `__len__` 就是告诉 Python 怎么数。


In [ ]:
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []

    def __len__(self):
        """返回商品件数"""
        return len(self.items)

# --- 执行过程 ---
# 第 9 行 c = Cart(["苹果", "香蕉", "牛奶"]):
#   ① self.items = ["苹果","香蕉","牛奶"]
# 第 10 行 n = len(c):
#   ① Python 看到 len(c),查找 c 的 __len__ 方法
#   ② 调用 c.__len__(),返回 len(self.items)
#   ③ len(self.items) = 3
#   ④ n = 3

c = Cart(["苹果", "香蕉", "牛奶"])
print(len(c))       # 输出: 3
print(c.__len__())  # 输出: 3 (等价写法)


> **逐行解剖**  
① `def __len__(self)` 必须返回**整数**  
② `len(c)` 等价于 `c.__len__()`  
③ 返回 0 表示空对象,Python 视其为**假值**  
④ `if cart:` 等价于 `if len(cart) > 0:`


**常见错误**

1. **返回非整数**:`return self.items` 返回列表,
   `len()` 会报 TypeError。
2. **忘记 self**:`def __len__():` 缺少 self 参数,
   调用时参数数量不对。
3. **返回负数**:`__len__` 返回负数会报 ValueError。


> 问自己:
> - `__len__` 必须返回什么类型?
> - `if cart:` 背后调用的是什么方法?
> - 空购物车 `len(cart)` 应该返回几?


**练习** —— 为 `Order` 添加 `__len__`

让 `len(order)` 返回订单的商品件数。


In [ ]:
# ============ 学员代码区 ============
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []
    def __len__(self):
        return len(self.items)

class Order:
    def __init__(self, cart):
        self.cart = cart
    # 请实现 __len__,返回商品件数

# cart = Cart(["书", "笔", "本"])
# order = Order(cart)
# print(len(order))  # 期望: 3


In [ ]:
# ============ 参考答案 ============
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []
    def __len__(self):
        return len(self.items)

class Order:
    def __init__(self, cart):
        self.cart = cart

    def __len__(self):
        # 委托给 cart 的 __len__
        return len(self.cart)

cart = Cart(["书", "笔", "本"])
order = Order(cart)
print(len(order))  # 输出: 3


#### __iter__ —— 让 for item in cart 遍历

> **痛点**  
`for item in cart` 报错,因为 Python 不知道
怎么"逐个取出"自定义对象的元素。

> **类比**  
`__iter__` 像给 Python 一个**迭代器**,
告诉它"每次取一个,取完为止"。

**两种实现方式**:
1. **yield 生成器**(简单):`__iter__` 里 yield 每个元素
2. **返回迭代器对象**(标准):`__iter__` 返回带 `__next__` 的对象


In [ ]:
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []

    def __iter__(self):
        """方式 1:用 yield 逐个返回元素"""
        for item in self.items:
            yield item

# --- 执行过程 ---
# 第 11 行 for item in c:
#   ① Python 调用 c.__iter__(),得到生成器
#   ② 每次循环调用 next(生成器)
#   ③ yield 返回下一个 item
#   ④ 没有元素时抛出 StopIteration,循环结束

c = Cart(["苹果", "香蕉", "牛奶"])
for item in c:
    print(item)
# 输出: 苹果
#       香蕉
#       牛奶


In [ ]:
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []

    def __iter__(self):
        """方式 2:返回列表的迭代器(更简洁)"""
        return iter(self.items)

# 直接委托给列表的 __iter__
c = Cart(["苹果", "香蕉"])
it = iter(c)       # 等价于 c.__iter__()
print(next(it))    # 输出: 苹果
print(next(it))    # 输出: 香蕉
# print(next(it))  # StopIteration

# --- 执行过程 ---
# 第 10 行 iter(c):
#   ① 调用 c.__iter__()
#   ② return iter(self.items) 返回列表迭代器
# 第 11 行 next(it):
#   ① 从迭代器取下一个元素
#   ② 返回 "苹果"


> **逐行解剖**  
① `yield item` 把 `__iter__` 变成**生成器函数**  
② `for x in obj` 等价于 `for x in obj.__iter__()`  
③ `return iter(self.items)` 直接委托,最简洁  
④ 有了 `__iter__`,也自动支持 `list(cart)`、`tuple(cart)`


**常见错误**

1. **返回列表而非迭代器**:`return self.items` 不报错,
   但语义不对,应该返回 `iter(self.items)`。
2. **忘记 yield**:`__iter__` 里写 `return item`,
   只返回一个元素就结束。
3. **只有 `__len__` 没有 `__iter__`**:`len()` 能跑,
   但 `for x in obj` 仍然报错。


> 问自己:
> - `for x in obj` 背后调用的是什么方法?
> - yield 和 return 在 `__iter__` 里有什么区别?
> - 为什么 `return iter(self.items)` 比 `return self.items` 好?


**练习** —— 让 `Order` 支持遍历商品

实现 `Order.__iter__`,让 `for item in order` 可用。


In [ ]:
# ============ 学员代码区 ============
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []
    def __iter__(self):
        return iter(self.items)

class Order:
    def __init__(self, cart):
        self.cart = cart
    # 请实现 __iter__

# cart = Cart(["书", "笔"])
# order = Order(cart)
# for item in order:
#     print(item)


In [ ]:
# ============ 参考答案 ============
class Cart:
    def __init__(self, items=None):
        self.items = items if items else []
    def __iter__(self):
        return iter(self.items)

class Order:
    def __init__(self, cart):
        self.cart = cart

    def __iter__(self):
        # 委托给 cart 的 __iter__
        return iter(self.cart)

cart = Cart(["书", "笔"])
order = Order(cart)
for item in order:
    print(item)
# 输出: 书
#       笔


#### __eq__ —— 同款判定(购物车去重)

> **痛点**  
两个商品属性完全相同,`==` 仍返回 False。
因为默认 `==` 比较的是**内存地址**(is),不是属性值。

> **类比**  
超市里两瓶同款矿泉水,**不是同一瓶**(is 不同),
但**是同款**(== 应该 True)。


In [ ]:
class Product:
    def __init__(self, sku, name, price):
        self.sku = sku      # 货号:唯一标识
        self.name = name
        self.price = price

    def __eq__(self, other):
        """SKU 相同 = 同款"""
        if not isinstance(other, Product):
            return NotImplemented
        return self.sku == other.sku

# --- 执行过程 ---
# 第 14 行 p1 = Product("A001", "矿泉水", 2.0):
#   ① 创建 Product 实例,sku="A001"
# 第 15 行 p2 = Product("A001", "矿泉水", 2.0):
#   ① 创建另一个 Product 实例,sku="A001"
# 第 16 行 p1 == p2:
#   ① Python 调用 p1.__eq__(p2)
#   ② self.sku == other.sku → "A001" == "A001" → True
# 第 17 行 p1 is p2:
#   ① 比较内存地址,两个不同对象 → False

p1 = Product("A001", "矿泉水", 2.0)
p2 = Product("A001", "矿泉水", 2.0)
p3 = Product("B002", "面包", 5.0)

print(p1 == p2)   # 输出: True  (同款)
print(p1 == p3)   # 输出: False (不同款)
print(p1 is p2)   # 输出: False (不是同一个对象)


> **逐行解剖**  
① `def __eq__(self, other)` 定义 `==` 行为  
② `isinstance` 检查类型,不同类型返回 NotImplemented  
③ `return self.sku == other.sku` 只比较 SKU  
④ `is` 永远比较地址,不受 `__eq__` 影响


**常见错误**

1. **忘记类型检查**:`return self.sku == other.sku`,
   当 other 不是 Product 时,访问 other.sku 会 AttributeError。
2. **返回 NotImplemented 而非 False**:类型不匹配时
   返回 NotImplemented,让 Python 尝试反向调用。
3. **只实现 `__eq__` 不实现 `__hash__`**:对象可比较,
   但不能放入 set/dict(后续讲)。


> 问自己:
> - `==` 和 `is` 有什么区别?
> - 为什么 `__eq__` 里要先检查类型?
> - 如果两个商品价格不同但 SKU 相同,算同款吗?


**练习** —— 同名同款判定

定义 `Product`,让 `__eq__` 比较 name(忽略 SKU)。
验证两同名商品 `==` 为 True。


In [ ]:
# ============ 学员代码区 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    # 请实现 __eq__,比较 name

# p1 = Product("苹果", 5.0)
# p2 = Product("苹果", 6.0)
# print(p1 == p2)  # 期望: True


In [ ]:
# ============ 参考答案 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.name == other.name

p1 = Product("苹果", 5.0)
p2 = Product("苹果", 6.0)
print(p1 == p2)  # 输出: True


#### __hash__ —— 支持放入 set 和 dict

> **痛点**  
实现了 `__eq__` 后,对象能比较了,
但放入 `set` 报错:unhashable type。

> **类比**  
`__hash__` 像给对象一个"身份证号",
set/dict 用它快速查找。同款商品应该有**相同**的身份证号。

**黄金法则**:如果 `a == b`,那么 `hash(a) == hash(b)`。


In [ ]:
class Product:
    def __init__(self, sku, name):
        self.sku = sku
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.sku == other.sku

    def __hash__(self):
        """用 SKU 的哈希值作为对象的哈希值"""
        return hash(self.sku)

# --- 执行过程 ---
# 第 17 行 products = {p1, p2, p3}:
#   ① 添加 p1:计算 hash("A001"),放入对应桶
#   ② 添加 p2:计算 hash("A001"),发现桶里已有
#   ③ 调用 __eq__ 比较:p1 == p2 → True,判定重复,不添加
#   ④ 添加 p3:hash("B002")不同,放入新桶
#   ⑤ 最终 set 只有 2 个元素

p1 = Product("A001", "矿泉水")
p2 = Product("A001", "矿泉水")
p3 = Product("B002", "面包")

products = {p1, p2, p3}
print(len(products))  # 输出: 2 (p1 和 p2 同款,去重)


> **逐行解剖**  
① `def __hash__(self)` 必须返回整数  
② `hash(self.sku)` 复用字符串的哈希函数  
③ 相等的对象必须有相等的哈希值(契约!)  
④ 只实现 `__eq__` 不实现 `__hash__`,对象不可哈希


**ASCII 内存图 —— set 去重原理**

```
set 内部结构:
┌─────────────────────────────┐
│ 哈希桶                       │
│  hash("A001") → [p1, p2]   │  ← __eq__ 判定重复
│  hash("B002") → [p3]       │
└─────────────────────────────┘

添加 p2 时:
  ① hash("A001") 找到桶 [p1]
  ② p1 == p2? → True → 不添加
```


**常见错误**

1. **只实现 `__eq__` 不实现 `__hash__`**:Python 自动把
   `__hash__` 设为 None,对象不可哈希。
2. **哈希值依赖可变属性**:用 `hash(self.price)`,
   改了 price 后哈希值变了,set 里找不到它。
3. **违反契约**:`a == b` 但 `hash(a) != hash(b)`,
   set/dict 行为错乱。


> 问自己:
> - 为什么 `__eq__` 和 `__hash__` 必须同时实现?
> - 如果两个对象相等,它们的哈希值必须满足什么条件?
> - 为什么不能用列表(可变)做哈希的依据?


**练习** —— 让 Product 支持 set 去重

实现 `__hash__`,让同名商品能去重。


In [ ]:
# ============ 学员代码区 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.name == other.name
    # 请实现 __hash__

# p1 = Product("苹果", 5.0)
# p2 = Product("苹果", 6.0)
# s = {p1, p2}
# print(len(s))  # 期望: 1


In [ ]:
# ============ 参考答案 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.name == other.name

    def __hash__(self):
        # 用 name 的哈希值(不可变字符串)
        return hash(self.name)

p1 = Product("苹果", 5.0)
p2 = Product("苹果", 6.0)
s = {p1, p2}
print(len(s))  # 输出: 1


#### __repr__ vs __str__ —— 面向开发者 vs 面向用户

> **痛点**  
`print(对象)` 打印 `<__main__.Product at 0x...>`,
调试时完全看不出对象里装了什么。

> **类比**  
- `__repr__` = 开发者的"X 光片":精确、无歧义
- `__str__` = 用户的"商品标签":友好、可读

**规则**:
- `repr(obj)` → 调用 `__repr__`,给开发者看
- `str(obj)` / `print(obj)` → 调用 `__str__`,给用户看
- 如果只实现 `__repr__`,`__str__` 默认用它


In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    def __repr__(self):
        """面向开发者:精确、能还原对象"""
        return f"Product(name={self.name!r}, price={self.price})"

    def __str__(self):
        """面向用户:友好可读"""
        return f"{self.name} ￥{self.price:.2f}"

# --- 执行过程 ---
# 第 16 行 p = Product("苹果", 5.0):
#   ① 创建实例,name="苹果",price=5.0
# 第 17 行 print(p):
#   ① Python 调用 str(p) → p.__str__()
#   ② 返回 "苹果 ￥5.00"
# 第 18 行 repr(p):
#  ① Python 调用 p.__repr__()
#   ② 返回 "Product(name='苹果', price=5.0)"

p = Product("苹果", 5.0)
print(p)          # 输出: 苹果 ￥5.00  (调用 __str__)
print(repr(p))    # 输出: Product(name='苹果', price=5.0)


In [ ]:
# 在列表中,元素显示用 __repr__
p1 = Product("苹果", 5.0)
p2 = Product("香蕉", 3.0)
print([p1, p2])
# 输出: [Product(name='苹果', price=5.0), Product(name='香蕉', price=3.0)]

# 如果只实现 __repr__,没有 __str__
class SimpleProduct:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return f"SimpleProduct({self.name!r})"

sp = SimpleProduct("梨")
print(sp)       # 输出: SimpleProduct('梨')  ← 用 __repr__ 兜底
print(repr(sp)) # 输出: SimpleProduct('梨')


> **逐行解剖**  
① `__repr__` 返回**能还原对象**的字符串(给开发者)  
② `__str__` 返回**用户友好**的字符串(给终端用户)  
③ `!r` 在 f-string 里调用 `repr(self.name)`,加引号  
④ 列表/字典显示元素时用 `__repr__`,不是 `__str__`


**常见错误**

1. **只实现 `__str__` 不实现 `__repr__`**:`repr(obj)` 仍显示
   默认地址,调试时不方便。建议**至少实现 `__repr__`**。
2. **返回非字符串**:`return self.price` 会 TypeError,
   必须返回 str。
3. **`__repr__` 太友好**:`__repr__` 应该精确,不要
   `"商品:苹果"` 这种,应该是 `Product(name='苹果')`。


> 问自己:
> - `print(obj)` 和 `repr(obj)` 分别调用什么方法?
> - 为什么建议至少实现 `__repr__`?
> - `__repr__` 的理想格式是什么?


**练习** —— 为 `Order` 添加 `__repr__`

让 `repr(order)` 显示订单的关键信息。


In [ ]:
# ============ 学员代码区 ============
class Order:
    def __init__(self, order_id, items):
        self.order_id = order_id
        self.items = items
    # 请实现 __repr__

# o = Order("ORD001", ["书", "笔"])
# print(repr(o))  # 期望: Order(order_id='ORD001', items=['书', '笔'])


In [ ]:
# ============ 参考答案 ============
class Order:
    def __init__(self, order_id, items):
        self.order_id = order_id
        self.items = items

    def __repr__(self):
        return (f"Order(order_id={self.order_id!r}, "
                f"items={self.items!r})")

o = Order("ORD001", ["书", "笔"])
print(repr(o))
# 输出: Order(order_id='ORD001', items=['书', '笔'])


#### 协议总结 —— 想支持什么操作就实现什么协议

> **核心认知**  
Python 的"魔法"都是**协议**。运算符、内置函数、
语法糖,背后都在查找对应的**双下划线方法**。

**常用协议速查表**:

| 你想支持 | 实现方法 | 触发方式 |
| --- | --- | --- |
| `a + b` | `__add__` | 加法运算 |
| `a == b` | `__eq__` | 相等比较 |
| `len(obj)` | `__len__` | 求长度 |
| `for x in obj` | `__iter__` | 迭代遍历 |
| `obj[k]` | `__getitem__` | 索引访问 |
| `print(obj)` | `__str__` | 用户输出 |
| `repr(obj)` | `__repr__` | 开发者输出 |
| `hash(obj)` | `__hash__` | set/dict 键 |
| `bool(obj)` | `__bool__` | 真假判断 |
| `a < b` | `__lt__` | 小于比较 |


**协议之间的关系**

```
┌─────────────────────────────────────────┐
│           让对象像内置类型一样自然         │
├─────────────┬───────────────────────────┤
│ 容器协议     │ __len_ _ + __iter_ _      │
│             │ → len() 和 for 循环        │
├─────────────┼───────────────────────────┤
│ 比较协议     │ __eq_ _ + __hash_ _       │
│             │ → == 和 set 去重           │
├─────────────┼───────────────────────────┤
│ 数值协议     │ __add_ _ / __sub_ _       │
│             │ → + / - 运算符             │
├─────────────┼───────────────────────────┤
│ 字符串协议   │ __str_ _ / __repr_ _      │
│             │ → print() / repr()         │
└─────────────┴───────────────────────────┘
```

**设计口诀**:先想"用户会怎么用这个对象",
再决定实现哪些协议。


> 问自己:
> - 如果想让 `obj[key]` 可用,要实现什么方法?
> - 如果想让对象能排序,要实现什么方法?
> - 为什么实现了 `__iter__` 就能用 `list(obj)`?


#### 综合练习 —— 电商订单系统 v2 (L1-L4 全员整合)

> **目标**  
整合 Day 08-11 全部知识,写一个可运行的系统。

**要求**:
1. **L1 封装**:`Product` 用 `@property` 保护 price
2. **L2 继承**:`Book` / `Food` 继承 `Product`
3. **L3 多态**:`describe()` 方法多态
4. **L4 组合+协议**:
   - `Cart` 组合多个 `Product`
   - `__add__` 合并购物车
   - `__len__` 返回件数
   - `__iter__` 遍历商品
   - `__eq__` 同款判定
   - `__repr__` 调试输出
   - `__hash__` 支持 set 去重


In [ ]:
# ============ 学员代码区 ============
import abc

class Product(abc.ABC):
    """抽象基类"""
    def __init__(self, sku, name, price):
        self.sku = sku
        self.name = name
        self.price = price  # 用 @property 保护
    # 请实现 @property + __eq__ + __hash__ + __repr__
    # 请实现 describe() 抽象方法

class Book(Product):
    """图书"""
    # 请实现 describe()

class Food(Product):
    """食品"""
    # 请实现 describe()

class Cart:
    """购物车"""
    # 请实现 __init__ + __add__ + __len__
    # 请实现 __iter__ + __repr__

# 测试代码(取消注释运行):
# b = Book("B001", "Python入门", 59.9)
# f = Food("F001", "面包", 8.5)
# c1 = Cart([b, f])
# c2 = Cart([b])
# c3 = c1 + c2
# print(f"共 {len(c3)} 件")
# for item in c3:
#     print(item.describe())


In [ ]:
# ============ 参考答案 ============
import abc

class Product(abc.ABC):
    """抽象基类"""
    def __init__(self, sku, name, price):
        self.sku = sku
        self.name = name
        self.price = price  # 触发 setter

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("价格不能为负")
        self._price = value

    def __eq__(self, other):
        if not isinstance(other, Product):
            return NotImplemented
        return self.sku == other.sku

    def __hash__(self):
        return hash(self.sku)

    def __repr__(self):
        return (f"{type(self).__name__}"
                f"(sku={self.sku!r}, name={self.name!r}, "
                f"price={self.price})")

    @abc.abstractmethod
    def describe(self):
        pass

class Book(Product):
    def describe(self):
        return f"图书《{self.name}》￥{self.price}"

class Food(Product):
    def describe(self):
        return f"食品[{self.name}] ￥{self.price}"

class Cart:
    def __init__(self, items=None):
        self.items = list(items) if items else []

    def __add__(self, other):
        if not isinstance(other, Cart):
            return NotImplemented
        return Cart(self.items + other.items)

    def __len__(self):
        return len(self.items)

    def __iter__(self):
        return iter(self.items)

    def __repr__(self):
        return f"Cart({self.items!r})"

# 测试
b = Book("B001", "Python入门", 59.9)
f = Food("F001", "面包", 8.5)
c1 = Cart([b, f])
c2 = Cart([b])
c3 = c1 + c2
print(f"共 {len(c3)} 件")
for item in c3:
    print(item.describe())
print(repr(c3))


#### 今日小结

| 概念 | 解决的痛点 | 关键方法 |
| --- | --- | --- |
| 组合 vs 继承 | has-a 不用继承 | 持有引用 |
| 继承三个坏味道 | 避免错误继承 | is-a 判断 |
| 组合设计 | 复杂对象拆简单 | 注入实例 |
| 运算符重载本质 | 运算符是语法糖 | `__add__` 等 |
| `__add__` | 合并两个对象 | 返回新对象 |
| `__len__` | len(obj) 可用 | 返回整数 |
| `__iter__` | for x in obj | yield/返回迭代器 |
| `__eq__` | == 比较属性 | 类型检查 |
| `__hash__` | 支持 set/dict | 与 __eq__ 一致 |
| `__repr__` vs `__str__` | 调试 vs 显示 | 至少实现 repr |
| 协议总结 | 统一认知 | 查表实现 |

**更多练习**

- 当堂练:course/lesson11/in_class/practice01-06.py
- 课后作业:course/lesson11/homework/task01-03.py
